In [ ]:
# Cell 1
!pip install -q transformers datasets evaluate accelerate thefuzz python-Levenshtein jiwer

import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate
from thefuzz import process

# Ensure GPU is being used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 100.9 MB/s eta 0:00:00
Using device: cuda


In [ ]:
!unzip -q prescription.zip -d /content/

In [ ]:
# Cell 2 (Updated to debug column names)
import pandas as pd

# Check headers first to ensure correct column mapping
train_df_sample = pd.read_csv("prescription/Training/training_labels.csv")
print("Training CSV Columns:", train_df_sample.columns.tolist())

class PrescriptionDataset(Dataset):
    def __init__(self, csv_file, img_dir, processor, max_target_length=128):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_target_length = max_target_length
        # Dynamically find the text column if 'MEDICINE' is missing
        self.text_col = 'MEDICINE' if 'MEDICINE' in self.df.columns else self.df.columns[1]
        self.img_col = 'IMAGE' if 'IMAGE' in self.df.columns else self.df.columns[0]
        print(f"Using columns: Image='{self.img_col}', Text='{self.text_col}' for {csv_file}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Load image
        img_name = str(self.df.iloc[idx][self.img_col])
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_name += '.png'

        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        # 2. Get OCR text
        text = str(self.df.iloc[idx][self.text_col])

        # 3. Process image
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        # 4. Tokenize text
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

# Re-initialize datasets with corrected logic
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-small-handwritten")

train_dataset = PrescriptionDataset(
    csv_file="prescription/Training/training_labels.csv",
    img_dir="prescription/Training/training_words",
    processor=processor
)

val_dataset = PrescriptionDataset(
    csv_file="prescription/Validation/validation_labels.csv",
    img_dir="prescription/Validation/validation_words",
    processor=processor
)

print(f"Training samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

Training CSV Columns: ['IMAGE', 'MEDICINE_NAME', 'GENERIC_NAME']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Using columns: Image='IMAGE', Text='MEDICINE_NAME' for prescription/Training/training_labels.csv
Using columns: Image='IMAGE', Text='MEDICINE_NAME' for prescription/Validation/validation_labels.csv
Training samples: 3120 | Validation samples: 780


In [ ]:
# Cell 3 (Updated with WER)
from transformers import GenerationConfig
import evaluate

model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-handwritten")
model.to(device)

# Configure model special tokens
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Set generation parameters via GenerationConfig
model.generation_config = GenerationConfig.from_pretrained("microsoft/trocr-small-handwritten")
model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
model.generation_config.eos_token_id = processor.tokenizer.sep_token_id
model.generation_config.max_length = 64
model.generation_config.early_stopping = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty = 2.0
model.generation_config.num_beams = 4

# Load BOTH metrics here
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

exact_match_metric = evaluate.load("exact_match")


def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Replace -100 with pad token so we can decode
    pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and truth
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    # Compute both CER and WER
    cer = cer_metric.compute(predictions=pred_str, references=labels_str)
    wer = wer_metric.compute(predictions=pred_str, references=labels_str)

    em = exact_match_metric.compute(predictions=pred_str, references=labels_str)

    return {"cer": cer, "wer": wer,
            "exact_match": em["exact_match"]}

pytorch_model.bin:   0%|          | 0.00/246M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

In [ ]:
# Cell 4
from transformers import default_data_collator

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr_prescription_model",
    predict_with_generate=True,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=4e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

# Start training
trainer.train()

# Save the final fine-tuned model
trainer.save_model("./trocr_prescription_final")
processor.save_pretrained("./trocr_prescription_final")
print("Model saved successfully!")

Step,Training Loss,Validation Loss,Cer,Wer,Exact Match
200,1.097864,0.858924,0.174089,0.360759,0.637179
400,0.301418,0.266707,0.114575,0.232911,0.764103
600,0.141428,0.151722,0.074291,0.150633,0.847436
800,0.128632,0.100633,0.062146,0.105063,0.893590
1000,0.064306,0.088017,0.051215,0.079747,0.919231
1200,0.026797,0.062909,0.028138,0.050633,0.948718
1400,0.019401,0.054830,0.031984,0.054430,0.944872
1600,0.007416,0.045835,0.019433,0.030380,0.969231
1800,0.000675,0.044446,0.019231,0.031646,0.967949
1950,0.008558,0.045051,0.021457,0.034177,0.965385


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
# 1. Build the Medicine-to-Generic Dictionary (STRICT NO-LEAKAGE VERSION)
df_train = pd.read_csv("prescription/Training/training_labels.csv")
df_val = pd.read_csv("prescription/Validation/validation_labels.csv")

# Only combine Training and Validation to build our knowledge base
all_data = pd.concat([df_train, df_val]).dropna(subset=['MEDICINE_NAME', 'GENERIC_NAME'])

# Create the mapping dictionary
medicine_to_generic = dict(zip(all_data['MEDICINE_NAME'], all_data['GENERIC_NAME']))
all_medicines = list(medicine_to_generic.keys())

print(f"Loaded {len(all_medicines)} unique written medicines into the dictionary from Train/Val sets only.")

# ... (The rest of the recognize_prescription function remains exactly the same)


# 2. Define the Inference Pipeline
def recognize_prescription(image_path):
    # Load and process image
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    # Generate OCR text
    generated_ids = model.generate(pixel_values)
    ocr_prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # STEP A: Fuzzy Match the OCR output to the closest known written MEDICINE
    best_medicine_match, confidence_score = process.extractOne(ocr_prediction, all_medicines)

    # STEP B: Look up the Generic Name using the perfectly matched Medicine name
    mapped_generic = medicine_to_generic.get(best_medicine_match, "Unknown")

    return {
        "Raw OCR Prediction": ocr_prediction,
        "Corrected Medicine Name": best_medicine_match,
        "Mapped Generic Name": mapped_generic,
        "Spelling Confidence": f"{confidence_score}%"
    }

# 3. Test it on an image from your Testing folder
test_image_path = "prescription/Testing/testing_words/700.png"
results = recognize_prescription(test_image_path)

print("\n--- Pipeline Results ---")
print(f"Raw OCR Prediction      : {results['Raw OCR Prediction']}")
print(f"Corrected Medicine Name : {results['Corrected Medicine Name']}")
print(f"Mapped Generic Name     : {results['Mapped Generic Name']}")
print(f"Spelling Confidence     : {results['Spelling Confidence']}")

Loaded 78 unique written medicines into the dictionary from Train/Val sets only.

--- Pipeline Results ---
Raw OCR Prediction      : Rozith
Corrected Medicine Name : Rozith
Mapped Generic Name     : Azithromycin Dihydrate
Spelling Confidence     : 100%


In [ ]:
# New Cell: Calculate Final Pipeline Exact Match Accuracy
df_test = pd.read_csv("prescription/Testing/testing_labels.csv")

total_images = len(df_test)
perfect_matches = 0

print(f"Testing {total_images} prescriptions...")

for index, row in df_test.iterrows():
    # 1. Get the image path and the true generic name
    img_name = str(row['IMAGE'])
    if not img_name.endswith('.png'): img_name += '.png'
    img_path = f"prescription/Testing/testing_words/{img_name}"

    true_generic = str(row['GENERIC_NAME'])

    # 2. Run your pipeline on the image
    try:
        results = recognize_prescription(img_path)
        predicted_generic = results['Mapped Generic Name']

        # 3. Check for exact match
        if predicted_generic == true_generic:
            perfect_matches += 1

    except Exception as e:
        print(f"Error processing {img_name}: {e}")

# 4. Calculate Final Accuracy
accuracy = (perfect_matches / total_images) * 100

print("\n=== FINAL SYSTEM REPORT ===")
print(f"Total Prescriptions Tested : {total_images}")
print(f"Perfect Generic Matches    : {perfect_matches}")
print(f"Exact Match Accuracy       : {accuracy:.2f}%")

Testing 780 prescriptions...

=== FINAL SYSTEM REPORT ===
Total Prescriptions Tested : 780
Perfect Generic Matches    : 754
Exact Match Accuracy       : 96.67%


In [ ]:
# New Cell: Calculate Final Testing CER and WER
import evaluate
import pandas as pd

# 1. Load the metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# 2. Load the Testing Data
df_test = pd.read_csv("prescription/Testing/testing_labels.csv")

# 3. Create empty lists to store all our predictions and answers
all_predictions = []
all_references = []

print(f"Evaluating OCR accuracy on {len(df_test)} test images...")

# 4. Loop through the test set
for index, row in df_test.iterrows():
    # Get image path
    img_name = str(row['IMAGE'])
    if not img_name.endswith('.png'): img_name += '.png'
    img_path = f"prescription/Testing/testing_words/{img_name}"

    # Get the TRUE handwritten text (the MEDICINE column)
    true_handwriting = str(row['MEDICINE_NAME'])

    try:
        # Run the image through your pipeline
        results = recognize_prescription(img_path)

        # Get the RAW prediction (Before fuzzy matching!)
        raw_prediction = results['Raw OCR Prediction']

        # Save both to our lists
        all_predictions.append(raw_prediction)
        all_references.append(true_handwriting)

    except Exception as e:
        print(f"Error processing {img_name}: {e}")

# 5. Calculate Final Metrics
# We pass the entire lists into the compute functions at once
final_cer = cer_metric.compute(predictions=all_predictions, references=all_references)
final_wer = wer_metric.compute(predictions=all_predictions, references=all_references)

# Print the Final Report
print("\n=== FINAL OCR VISION REPORT (TESTING SET) ===")
print(f"Total Images Evaluated : {len(all_predictions)}")
print(f"Testing CER            : {final_cer:.4f} ({final_cer * 100:.2f}%)")
print(f"Testing WER            : {final_wer:.4f} ({final_wer * 100:.2f}%)")

Evaluating OCR accuracy on 780 test images...

=== FINAL OCR VISION REPORT (TESTING SET) ===
Total Images Evaluated : 780
Testing CER            : 0.0512 (5.12%)
Testing WER            : 0.0709 (7.09%)
